# Workshop: PII Detection, Sanitization, and Use

A walkthrough that goes end-to-end: **detect** sensitive data, **sanitize** it under different rewrite modes, and put the sanitized text to **use** in a downstream search task. Default detectors enabled here are Presidio, GLiNER, NVIDIA's GLiNER variant, and OPF — plus optional Skyflow Detect if you have credentials.

**What this notebook does:**
1. **Detection**: installs the harness, materializes a deterministic sample fixture from one of five ai4privacy datasets, runs each selected detector, and renders a Markdown report with results.
2. **Sanitization**: replaces every detected span under four modes — `redact` (`********`), `label` (`[EMAIL]`), `label_number` (`[EMAIL_1]`), and `label_token` (`[EMAIL_jRc7QGn]`, deterministic vault token) — side-by-side per detector.
3. **Use**: a small BM25 search demo. Sanitizes a small corpus and a pair of queries under every mode, indexes each, and shows precision/recall per mode against known gold relevance sets. Demonstrates that only `label_token` preserves referential integrity.
4. **What's next** — where you can take this next.


## 1. Setup: install the testing harness and dependencies

Run these cells once per Colab session. Installs:
- spaCy models for Presidio.
- The OPF source repo (open-weight PII detector from OpenAI)
- The comparison harness (this repo's `eval/` package)
- GLiNER, Presidio, and supporting libs


In [ ]:
# spaCy models for Presidio. en_core_web_lg is required; the others are
# optional and only needed if you want multilingual Presidio (skip if running
# default English-only Presidio — most cases).
#
# IMPORTANT (Colab): running this cell upgrades numpy/typing-extensions and
# Colab will pop up "RESTART SESSION" when it finishes. Click restart, then
# use Runtime → Run all to continue. Doing the spaCy install first means the
# slow harness install in the next cell only happens once (after the restart),
# rather than getting wiped out and re-run.
!python -m spacy download en_core_web_lg -q

# Uncomment the next 5 lines for multilingual Presidio (~3 GB extra download):
# !python -m spacy download nl_core_news_lg -q
# !python -m spacy download fr_core_news_lg -q
# !python -m spacy download de_core_news_lg -q
# !python -m spacy download it_core_news_lg -q
# !python -m spacy download es_core_news_lg -q

print("\nspaCy models ready. If Colab prompts to restart the session, do it now and re-run from the top.")


In [ ]:
HARNESS_REPO = "https://github.com/jstjoe/local-privacy.git"
HARNESS_BRANCH = "main"   # set to a branch name (e.g. "jstjoe/notebook-iteration") to test a PR before merge
OPF_REPO = "https://github.com/openai/privacy-filter.git"

import os, subprocess, sys

# Triton has no stable Apple Silicon support and isn't needed on Colab CPU/GPU.
# Setting before any opf import keeps the runtime on the vanilla PyTorch MoE path.
os.environ.setdefault("OPF_MOE_TRITON", "0")

PIP = f"{sys.executable} -m pip"  # ensure we install into the notebook's kernel

def _run(cmd, *, msg, show_output=False):
    print(f"==> {msg}")
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.returncode != 0 or show_output:
        if r.stdout: print(r.stdout)
        if r.stderr: print(r.stderr)
    if r.returncode != 0:
        raise RuntimeError(f"{msg} failed (exit {r.returncode}). See output above.")

if not os.path.exists("/content/privacy-filter"):
    _run(f"git clone --depth 1 {OPF_REPO} /content/privacy-filter", msg="clone privacy-filter")
if not os.path.exists("/content/local-privacy"):
    _run(f"git clone --depth 1 --branch {HARNESS_BRANCH} {HARNESS_REPO} /content/local-privacy", msg=f"clone local-privacy@{HARNESS_BRANCH} (must be public, or use a PAT in HARNESS_REPO)")

# Pip-install for dependency resolution (torch, datasets, gliner, presidio, etc).
# Whether or not pip places the top-level packages in site-packages reliably on
# Colab, we also add the source directories to sys.path below so imports always
# work.
_run(f"{PIP} install -q /content/privacy-filter", msg="install opf + deps")
_run(f"{PIP} install -q /content/local-privacy/eval", msg="install opf-eval + deps")
_run(f"{PIP} install -q /content/local-privacy/api", msg="install opf-api + deps (needed for Sanitization → label_token mode)")
_run(f"{PIP} install -q rank-bm25", msg="install rank-bm25 (used by the Use section to search sanitized text)")

# Belt-and-suspenders: expose the source trees on sys.path so `import opf` and
# `import opf_eval` resolve regardless of what pip did with the editable hooks.
for src_dir in ("/content/privacy-filter", "/content/local-privacy/eval/src", "/content/local-privacy/api/src"):
    if src_dir not in sys.path:
        sys.path.insert(0, src_dir)

# Confirm the kernel can import everything we need.
import importlib
for mod in ("opf", "opf_eval", "opf_eval.runner", "opf_eval.report", "opf_eval.fixtures", "opf_eval.transforms", "opf_api", "opf_api.vault_tokens", "rank_bm25"):
    importlib.import_module(mod)
    print(f"  ok: {mod}")

print("\nsetup complete.")


## 2. Imports and configuration

Pick the dataset, sample size, and detectors to benchmark in the cell below. Or just click Run.

**Datasets:**
- `pii_masking_300k` (default), `pii_masking_200k`, `pii_masking_400k` — legacy ai4privacy variants (different vocabularies)
- `openpii_nano` (1k), `openpii_mini` (10k) — OpenPII vocabulary

**Available detector reference:**
- `opf` — OpenAI Privacy Filter (open-weight, local) — overall local leader
- `gliner` — default GLiNER multilingual PII (`urchade/gliner_multi_pii-v1`) — prompts auto-restricted to dataset vocab
- `gliner_nvidia` — Nvidia gliner-PII on `urchade/gliner_large-v2.1` (570M base, threshold 0.3, NVIDIA Open Model License) — strongest of the GLiNER variants
- `gliner_gretel_small` / `gliner_gretel_large` — Gretel bi-encoder GLiNER (threshold 0.7, English-only training, snake_case 41-label vocab)
- `ai4privacy_modernbert` — ai4privacy ModernBERT-base (~150M, MIT, 8 languages, OpenPII vocab)
- `openmed` — OpenMed PII via `openmed.extract_pii(lang=…)` — DeBERTa-based per-language models, snake_case 55-label vocab
- `presidio` — Microsoft Presidio English-only (regex + NER, local)
- `presidio_multilang` — Presidio with all 6 spaCy models (requires the optional downloads above)
- `skyflow` — Skyflow Detect API; `entity_types` auto-derived from dataset canonicals (requires creds)
- `skyflow_full` — Skyflow Detect API with all ~70 entity types (requires creds)

In [ ]:
from pathlib import Path
from IPython.display import Markdown, display

import torch
from opf_eval import fixtures, runner, report

# === EDIT THESE ===
DATASET = "pii_masking_200k"  # one of: pii_masking_300k, pii_masking_200k, pii_masking_400k, openpii_nano, openpii_mini
N_EXAMPLES = 100              # 100 for a quick smoke test, 1000 for a real bench, 5000 for stable signal
# Pick from any of these (eval-side runner names):
#   opf
#   gliner, gliner_nvidia, gliner_gretel_small, gliner_gretel_large
#   ai4privacy_modernbert, openmed
#   presidio, presidio_multilang
#   skyflow, skyflow_full        # add these after setting credentials below
DETECTORS = ["presidio", "gliner", "gliner_nvidia", "opf"]
FIXTURE_SEED = 42             # deterministic sample
RUN_NAME = "colab_demo"       # used as the output dir
# ===================

# Device: cuda > mps > cpu. To force CPU, set DEVICE = "cpu".
# Colab GPU runtime: Runtime -> Change runtime type -> T4 / L4 / A100 GPU.
# OPF speeds up ~10×, gliner_nvidia ~10×, others ~3-5× on T4 vs CPU.
DEVICE = (
    "cuda" if torch.cuda.is_available()
    else "mps" if hasattr(torch.backends, "mps") and torch.backends.mps.is_available()
    else "cpu"
)

FIXTURES_PATH = Path(f"/content/data/{DATASET}_{N_EXAMPLES}.jsonl")
OUT_DIR = Path(f"/content/results/{RUN_NAME}")

FIXTURES_PATH.parent.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"device:                 {DEVICE}")
print(f"dataset:                {DATASET}")
print(f"will write fixtures to: {FIXTURES_PATH}")
print(f"will write results to:  {OUT_DIR}")
print(f"detectors:              {DETECTORS}")

## 3. (Optional) Skyflow credentials

Auto-skips the credential setup below if `skyflow` (or `skyflow_full`) isn't in your `DETECTORS` list — so you can just run all cells in order without curating which to skip.

If you *are* benchmarking Skyflow, add these to **Colab Secrets** (key icon in the left sidebar):
- `SKYFLOW_VAULT_URL` — e.g. `https://abc123.vault.skyflowapis.com`
- `SKYFLOW_VAULT_ID` — vault UUID
- `SKYFLOW_BEARER_TOKEN` — short-lived bearer token


In [ ]:
_skyflow_in_use = bool({"skyflow", "skyflow_full"} & set(DETECTORS))
if not _skyflow_in_use:
    print("No skyflow detector in DETECTORS — skipping credential setup.")
else:
    try:
        from google.colab import userdata
        for var in ("SKYFLOW_VAULT_URL", "SKYFLOW_VAULT_ID", "SKYFLOW_BEARER_TOKEN"):
            try:
                os.environ[var] = userdata.get(var)
            except Exception:
                print(f"  {var}: not set in Colab Secrets")
        if all(v in os.environ for v in ("SKYFLOW_VAULT_URL", "SKYFLOW_VAULT_ID", "SKYFLOW_BEARER_TOKEN")):
            print("Skyflow creds loaded.")
        else:
            print("Skyflow creds incomplete — drop the skyflow detector from DETECTORS or fix the secrets.")
    except ImportError:
        print("Not in Colab. Set SKYFLOW_VAULT_URL / SKYFLOW_VAULT_ID / SKYFLOW_BEARER_TOKEN in your shell env instead.")


## 4. Create the sample data subset

Materialize the test set: a fixed sample from the chosen dataset, with the gold PII spans the detectors will be scored against. First run downloads the dataset shards (~700 MB); cached after that.


In [ ]:
if not FIXTURES_PATH.exists():
    n = fixtures.materialize(FIXTURES_PATH, N_EXAMPLES, dataset=DATASET, seed=FIXTURE_SEED)
    print(f"wrote {n} examples to {FIXTURES_PATH}")
else:
    print(f"reusing existing fixtures at {FIXTURES_PATH}")

import json
with FIXTURES_PATH.open() as f:
    sample = json.loads(f.readline())
print(f"\nfirst fixture keys: {list(sample.keys())}")
print(f"first fixture preview: {sample['text'][:120]}...")
print(f"first fixture gold spans: {len(sample['gold_spans'])} spans")

## 5. Run the detectors

Runs each selected detector over every example in the fixture set you just materialized. One JSONL of predictions per detector is streamed to `OUT_DIR`, ready for the report cell below.


In [ ]:
runner.run(
    fixtures=FIXTURES_PATH,
    detector_names=DETECTORS,
    out_dir=OUT_DIR,
    dataset=DATASET,             # threads dataset_canonicals into skyflow + gliner builders
    device=DEVICE,               # cuda/mps/cpu — auto-detected in cell 5
    skyflow_workers=1,           # serial; bump if your Skyflow plan allows
    skyflow_min_interval_ms=0,   # add throttling if rate-limited
)

print("\nfiles written:")
for p in sorted(OUT_DIR.iterdir()):
    print(f"  {p.name}")

## 6. Display the report

Scores every detector's predictions against the gold spans and renders the result inline — headline F1, per-category F1, and per-language F1 across the strict / exact / partial / type SemEval schemas.


In [ ]:
md = report.build_report(OUT_DIR, FIXTURES_PATH)
(OUT_DIR / "report.md").write_text(md)

display(Markdown(md))

## 7. (Optional) Visualizations

Bar charts of per-category F1 and per-detector latency. Use when you want a chart instead of a table.


In [ ]:
import json, statistics
import matplotlib.pyplot as plt
import numpy as np

from opf_eval.datasets import get as get_dataset_config
from opf_eval.taxonomy import dataset_canonicals
from opf_eval.nervaluate_metrics import score as semeval_score

fixture_records = [json.loads(l) for l in FIXTURES_PATH.open() if l.strip()]
fixture_index = {r["id"]: r for r in fixture_records}

# Use the chosen dataset's annotated canonicals as the chart x-axis.
vocab_key = get_dataset_config(DATASET).vocab_key
labels = sorted(dataset_canonicals(vocab_key))
print(f"chart labels ({len(labels)}): {labels}")

def detector_data(name):
    """Run nervaluate against this detector's raw output. Returns
    (per-tag F1 dict, latencies). Same Type-schema F1 as the report's
    per-category section."""
    path = OUT_DIR / f"raw_{name}.jsonl"
    if not path.exists():
        return None
    records = [json.loads(l) for l in path.open() if l.strip()]
    pairs = []
    latencies = []
    for r in records:
        if r.get("error"):
            continue
        gold = fixture_index[r["id"]]["gold_spans"]
        # Filter both sides to the dataset's annotated canonicals.
        pred = [s for s in r["spans"] if s["label"] in set(labels)]
        gold = [s for s in gold if s["label"] in set(labels)]
        pairs.append((pred, gold))
        latencies.append(r["latency_ms"])
    sem = semeval_score(detector=name, pairs=pairs, tags=labels)
    by_label_f1 = {
        lbl: sem.by_label.get(lbl, {}).get("ent_type", {}).get("f1", 0.0)
        for lbl in labels
    }
    return by_label_f1, latencies

# Auto-discover every detector with a raw_*.jsonl in OUT_DIR.
all_detectors = sorted(p.stem.removeprefix("raw_") for p in OUT_DIR.glob("raw_*.jsonl"))
results = {d: detector_data(d) for d in all_detectors if detector_data(d)}
print(f"detectors in this run: {list(results)}")

# === Per-category F1 bar chart (SemEval Type schema) ===
x = np.arange(len(labels))
bar_w = 0.8 / max(len(results), 1)
fig, ax = plt.subplots(figsize=(max(12, 0.9 * len(labels)), 5))
for i, (det, (f1s_dict, _)) in enumerate(results.items()):
    f1s = [f1s_dict.get(lbl, 0.0) for lbl in labels]
    ax.bar(x + i * bar_w, f1s, bar_w, label=det)
ax.set_xticks(x + bar_w * (len(results) - 1) / 2)
ax.set_xticklabels(labels, rotation=20)
ax.set_ylabel("SemEval Type F1 (any overlap + matching label)")
ax.set_title(f"Per-category F1 by detector ({DATASET})")
ax.set_ylim(0, 1)
ax.legend(loc="upper right", ncol=2 if len(results) > 4 else 1, fontsize=8)
ax.grid(axis="y", linestyle=":", alpha=0.5)
plt.tight_layout()
plt.show()


In [ ]:
# === Latency comparison ===
fig, ax = plt.subplots(figsize=(10, 5))
names = list(results.keys())
p50s = [statistics.median(results[n][1]) for n in names]
p95s = [sorted(results[n][1])[int(0.95 * (len(results[n][1]) - 1))] for n in names]
p99s = [sorted(results[n][1])[int(0.99 * (len(results[n][1]) - 1))] for n in names]
x = np.arange(len(names))
ax.bar(x - 0.25, p50s, 0.25, label="p50")
ax.bar(x, p95s, 0.25, label="p95")
ax.bar(x + 0.25, p99s, 0.25, label="p99")
ax.set_xticks(x)
ax.set_xticklabels(names, rotation=15)
ax.set_ylabel("latency (ms)")
ax.set_title("Per-detector latency")
ax.set_yscale("log")
ax.legend()
ax.grid(axis="y", linestyle=":", alpha=0.5, which="both")
plt.tight_layout()
plt.show()

## 8. (Optional) Saving the run

Colab storage is ephemeral. To keep results, mount Drive and copy the run dir.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import shutil
dest = "/content/drive/MyDrive/pii_benchmark_runs/" + OUT_DIR.name
shutil.copytree(OUT_DIR, dest, dirs_exist_ok=True)
print(f"copied {OUT_DIR} -> {dest}")


# Sanitization

Detection (sections 1–8 above) tells you _where_ sensitive data is. Sanitization is what you do with it. The cells below show, side-by-side on the same fixtures, what each of the three available transform modes produces — from plain `[LABEL]` placeholders all the way to deterministic Skyflow vault tokens that survive across documents.


## 1. Setup and configuration

The benchmark above scores *detection* quality (did the detector find the right spans?). The cells below answer a different question: **once you've found the spans, what does the transformed text actually look like?**

Four sanitization modes are demonstrated side-by-side on the same fixtures the detectors above ran against:

| Mode | Looks like | What it preserves |
|---|---|---|
| `redact` | `********` | Nothing — fixed-length asterisks regardless of original span length. |
| `label` | `[EMAIL]` | Category only. All Alices and all Bobs look identical. |
| `label_number` | `[EMAIL_1]` | Identity **within one document**: same value gets the same number; different values get different numbers. |
| `label_token` | `[EMAIL_jRc7QGn]` | Identity **across documents and time** via a Skyflow vault. Deterministic — the same plaintext maps to the same token forever, regardless of which detector found it. |

Edit the cell below to change the demo size or detector list. Sections 2–4 reuse the `raw_<detector>.jsonl` files from section 5 of the benchmark — no detector re-runs.


In [ ]:
# === EDIT THESE ===
DEMO_N_EXAMPLES = 6                         # how many fixtures to display (kept small — qualitative)
DEMO_DETECTORS = DETECTORS                  # reuses whatever cell 5 had
DEMO_MODES = ["redact", "label", "label_number", "label_token"]
# ===================

print(f"will render {DEMO_N_EXAMPLES} examples × {len(DEMO_DETECTORS)} detectors × {len(DEMO_MODES)} modes")
print(f"detectors: {DEMO_DETECTORS}")
print(f"modes:     {DEMO_MODES}")

## 2. (Optional) Token-vault credentials — required for `label_token` mode

`label_token` mode rewrites every detected value into a deterministic 7-character token backed by a Skyflow vault. The same plaintext always gets the same token — across documents, across requests, across detectors. That's the property the Use section depends on; without it, search across sanitized text doesn't work.

Operator one-time setup is documented in [docs/token-vault-setup.md](https://github.com/jstjoe/local-privacy/blob/main/docs/token-vault-setup.md). Once the vault exists, set three Colab secrets: `SKYFLOW_TOKEN_VAULT_URL`, `SKYFLOW_TOKEN_VAULT_ID`, and `SKYFLOW_TOKEN_BEARER_TOKEN` (or reuse `SKYFLOW_BEARER_TOKEN` from section 3).


In [ ]:
import os
from opf_api.vault_tokens import TokenVaultClient

try:
    from google.colab import userdata  # type: ignore[import-not-found]
    for key in ("SKYFLOW_TOKEN_VAULT_URL", "SKYFLOW_TOKEN_VAULT_ID", "SKYFLOW_TOKEN_BEARER_TOKEN"):
        try:
            os.environ.setdefault(key, userdata.get(key) or "")
        except Exception:
            pass
except ImportError:
    pass  # not on Colab — env should already be set

TOKEN_VAULT = TokenVaultClient.from_env()
if TOKEN_VAULT is None:
    print("token vault not configured — `label_token` column will show a placeholder.")
    print("set SKYFLOW_TOKEN_VAULT_URL + _ID + a bearer to enable it.")
else:
    print("token vault configured — `label_token` column will show real 7-char tokens.")

## 3. Load fixtures and detector predictions

Pull a handful of fixtures and the corresponding predictions from every selected detector. Small on purpose — this section is a qualitative side-by-side, not another benchmark.


In [ ]:
# Load fixtures + each detector's predictions from disk.
import json

with FIXTURES_PATH.open() as f:
    all_fixtures = [json.loads(line) for line in f]

demo_fixtures = all_fixtures[:DEMO_N_EXAMPLES]

predictions = {}  # {detector_name: {fixture_id: [span, ...]}}
for detector in DEMO_DETECTORS:
    path = OUT_DIR / f"raw_{detector}.jsonl"
    if not path.exists():
        print(f"skipping {detector}: {path.name} not found — run cell 11 first")
        continue
    by_id = {}
    with path.open() as f:
        for line in f:
            row = json.loads(line)
            by_id[row["id"]] = row.get("spans") or []
    predictions[detector] = by_id

print(f"loaded predictions for: {sorted(predictions)}")
print(f"first demo fixture id: {demo_fixtures[0]['id']}, text preview: {demo_fixtures[0]['text'][:80]}...")

## 4. Render the results

One markdown table per example: rows are detectors, columns are sanitization modes. Read across a row to see what each mode does to the same set of detected spans. Read down a column to see how detectors disagree under one mode.


In [ ]:
from IPython.display import Markdown, display
from opf_eval.transforms import render_modes

def _esc(s: str) -> str:
    return s.replace("|", "\\|").replace("\n", " ")

for fx in demo_fixtures:
    fid = fx["id"]
    text = fx["text"]
    parts = [f"### Example `{fid}`\n", f"> {_esc(text)}\n"]
    header = "| detector | " + " | ".join(DEMO_MODES) + " |"
    sep = "|" + "|".join(["---"] * (len(DEMO_MODES) + 1)) + "|"
    parts.append(header)
    parts.append(sep)
    for detector, by_id in predictions.items():
        spans = by_id.get(fid, [])
        rendered = render_modes(
            text, spans,
            modes=DEMO_MODES,
            token_vault_client=TOKEN_VAULT,
        )
        row = [detector] + [_esc(rendered.get(m, "")) for m in DEMO_MODES]
        parts.append("| " + " | ".join(row) + " |")
    display(Markdown("\n".join(parts)))

**Reading the rows.**

- Identical label and label_number columns mean the detector found only one entity in that example.
- `label_token` shows a `(set SKYFLOW_TOKEN_VAULT_* to enable)` placeholder when the vault from section 2 above is unconfigured. Spans whose canonical label has no vault column (rare) fall back to `[LABEL]` per span.
- Different detectors disagree on span boundaries and labels — that's the same disagreement the F1 scores in section 6 quantify, surfaced here visually.


# Use

Detection finds where PII is. Sanitization rewrites it. But downstream tools — search, retrieval, RAG, joins — operate on the *rewritten* text. The sanitization mode you pick determines what those tools can still do.

The cells below stand up a tiny BM25 search demo: twenty sample documents, two natural-language queries that ask about specific email addresses, and a side-by-side comparison of retrieval quality across each sanitization mode. Only `label_token` preserves the property — same plaintext maps to the same token across separate documents — that search actually needs.


## 1. Setup and configuration
Set up the search corpus and the queries we'll evaluate. Each query carries its gold relevance set — the doc indices that *should* come back — so section 4 can score every mode's retrieval against the right answer.


In [ ]:
# Edit these to drop in your own corpus, queries, and gold relevance.
# Each query carries its gold-relevant doc ids so section 4 can compute
# precision / recall and flag every retrieved hit with ✓ or ✗.

# Mix of PII categories (EMAIL, PHONE, ADDRESS, PERSON) and PII-free
# fillers. The multi-email docs are deliberately ordered so that the
# same person (alice) appears as the first email in some docs and the
# second email in others — that's what breaks `label_number`: the
# query's `[EMAIL_1]` matches both the right alice docs AND the wrong
# bob/charlie/etc. docs whose first email happens to be a different
# person, and misses the doc where alice is `[EMAIL_2]`.

DEMO_USE_CORPUS = [
    # === PII-free fillers ===
    "Project status: design phase nearly complete.",                          # 0
    "The annual review covered Q3 earnings projections.",                     # 1
    "Team standup notes from Wednesday's meeting.",                           # 2
    "Quarterly OKRs were aligned to product strategy.",                       # 3
    "Roadmap discussion postponed to next sprint.",                           # 4

    # === Alice — single-email docs ===
    "Alice (alice@example.com) drafted the proposal last quarter.",           # 5
    "The team lead is alice@example.com per the org chart.",                  # 6

    # === Alice — multi-email; in doc 7 alice is the SECOND email,
    #     so label_number marks her [EMAIL_2] there instead of [EMAIL_1]. ===
    "Bob (bob@example.com) emailed alice@example.com about the file.",        # 7
    "Forwarded from alice@example.com to ops@example.com on Friday.",         # 8

    # === Charlie — single-email docs ===
    "Charlie owns the budget account charlie@example.com.",                   # 9
    "Don submitted feedback to charlie@example.com about scope.",             # 10

    # === Phone-only docs (no email, no address). Includes a multi-phone
    #     doc to deliberately outrank single-email docs under `redact`. ===
    "Voicemails from +1-628-555-0199, +1-510-555-0143, and +1-415-555-0100 logged.",  # 11  3 phones
    "On-call rotation: +1-628-555-0199 backup, +1-510-555-0143 escalation.",  # 12  2 phones
    "Hotline +1-415-555-0100 routed to escalations queue.",                   # 13  1 phone

    # === Address-only docs (no email, no phone). Same trick — one
    #     multi-address doc to confuse `redact` rankings. ===
    "Address change form: 123 Main St to 456 Pine Ave then 789 Oak Rd.",      # 14  3 addresses
    "Frank relocated to 456 Pine Ave last month for the move.",               # 15  1 address
    "Office HQ at 789 Oak Rd opens after renovation.",                        # 16  1 address

    # === Multi-PII non-alice (email + phone in one doc) ===
    "Sales contact: jay@example.com or call +1-415-555-0100.",                # 17

    # === Email-only, non-alice / non-charlie ===
    "Compliance review thread between ops@example.com and counsel.",          # 18
    "Newsletter signup from rachel@example.com confirmed.",                   # 19
]

# Short, keyword-style queries — closer to what a user types in a search
# bar. The PII value itself drives the ranking, so the modes show their
# real discrimination instead of being dominated by common words like
# "find" / "documents" that don't appear in the corpus.
DEMO_USE_QUERIES = [
    {
        "text": "alice@example.com",
        "relevant_doc_ids": {5, 6, 7, 8},
    },
    {
        "text": "charlie@example.com",
        "relevant_doc_ids": {9, 10},
    },
]

DEMO_USE_TOP_K = 5

# Which detector runs over both the corpus and the queries. Corpus +
# query MUST share one sanitizer or the modes won't line up. None =
# reuse the same default the Sanitization section picked (presidio if
# available — fast and CPU-only).
DEMO_USE_DETECTOR = None

print(f"corpus: {len(DEMO_USE_CORPUS)} docs, queries: {len(DEMO_USE_QUERIES)}, top_k: {DEMO_USE_TOP_K}")


## 2. Sanitize the corpus and queries

Sanitize both sides of the search — the documents *and* the queries — under every mode. Both sides must go through the same renderer (and, for `label_token`, the same vault), otherwise the tokens won't line up. `label_token` is skipped here if the vault from Sanitization section 2 wasn't configured.

We also keep a `plain` (unsanitized) row as the gold-standard reference — what retrieval looks like with no privacy layer at all.


In [ ]:
from opf_eval.runner import _build_detector
from opf_eval.taxonomy import CANONICAL_LABELS
from opf_eval.transforms import render_modes

# Corpus + queries share one sanitizer — modes only line up when
# both sides go through the same renderer.
use_det_name = DEMO_USE_DETECTOR
if use_det_name is None:
    if "presidio" in DEMO_DETECTORS:
        use_det_name = "presidio"
    elif DEMO_DETECTORS:
        use_det_name = DEMO_DETECTORS[0]
    else:
        use_det_name = "presidio"

print(f"sanitizing corpus + queries with: {use_det_name}\n")
use_det = _build_detector(
    use_det_name,
    dataset_canonicals_set=set(CANONICAL_LABELS),
    device=DEVICE,
)

# Modes to compare. "plain" is the no-sanitization reference — the
# upper bound of retrieval quality. label_token is only included if
# the token vault is configured (section 2 of Sanitization).
USE_MODES = ["plain", "redact", "label", "label_number"]
if TOKEN_VAULT is not None:
    USE_MODES.append("label_token")
else:
    print("(label_token mode skipped — TOKEN_VAULT not configured)")

_active = [m for m in USE_MODES if m != "plain"]

def _sanitize_under(text):
    spans = use_det.detect(text).get("spans") or []
    rendered = render_modes(text, spans, modes=_active, token_vault_client=TOKEN_VAULT)
    rendered["plain"] = text
    return rendered

corpus_by_mode = {m: [] for m in USE_MODES}
for doc in DEMO_USE_CORPUS:
    r = _sanitize_under(doc)
    for m in USE_MODES:
        corpus_by_mode[m].append(r[m])

queries_by_mode = {m: [] for m in USE_MODES}
for q in DEMO_USE_QUERIES:
    r = _sanitize_under(q["text"])
    for m in USE_MODES:
        queries_by_mode[m].append(r[m])

# Preview so it's obvious what got stored under each mode.
print("corpus doc 0 by mode:")
for m in USE_MODES:
    print(f"  {m:14s} {corpus_by_mode[m][0]}")
print("\nquery 0 by mode:")
for m in USE_MODES:
    print(f"  {m:14s} {queries_by_mode[m][0]}")


## 3. Index documents and run each sanitized query

Build one search index per mode and run each query through. The retrieval results — top-K document indices per (mode, query) — feed the comparison table in the next section.


In [ ]:
import re
from rank_bm25 import BM25Okapi

# Crude tokenizer: lowercase, whitespace-split, strip leading + trailing
# common punctuation (parens, brackets, sentence terminators). The key
# property: the *core* of a bracketed token like [EMAIL_u8UBDWQ] survives
# as one indexable string ("email_u8ubdwq") in both docs and queries,
# regardless of surrounding punctuation, so BM25 lines them up exactly.
_STRIP = re.compile(r"^[\[\](){}<>.,;:!?'\"]+|[\[\](){}<>.,;:!?'\"]+$")

def _tok(text):
    out = []
    for w in text.lower().split():
        w = _STRIP.sub("", w)
        if w:
            out.append(w)
    return out

# One BM25 index per (mode, corpus). Each query runs against the index
# that was built under the same mode it itself was sanitized under.
bm25_by_mode = {
    m: BM25Okapi([_tok(doc) for doc in corpus_by_mode[m]])
    for m in USE_MODES
}

# Record the top-k doc indices per (query, mode). We rank by raw BM25
# score and take the top k — no positive-score filter, since on a tiny
# corpus a term in >half the docs gets a negative IDF and the
# right-on-topic hits would be dropped.
retrievals = {}
for qi in range(len(DEMO_USE_QUERIES)):
    for m in USE_MODES:
        scores = bm25_by_mode[m].get_scores(_tok(queries_by_mode[m][qi]))
        ranked = sorted(range(len(scores)), key=scores.__getitem__, reverse=True)
        retrievals[(qi, m)] = ranked[:DEMO_USE_TOP_K]

print(f"indexed {len(USE_MODES)} modes × {len(DEMO_USE_CORPUS)} docs")
print(f"ran     {len(USE_MODES)} modes × {len(DEMO_USE_QUERIES)} queries")


## 4. Compare retrieval across modes

Now score each mode against the gold relevance set. One table per query: every row shows the sanitized query, the top-`k` retrievals (✓ if in the gold set, ✗ if not), and the resulting precision and recall.


In [ ]:
from IPython.display import Markdown, display

def _esc_cell(text):
    return text.replace("|", "\\|").replace("\n", " ")

parts = []
for qi, q in enumerate(DEMO_USE_QUERIES):
    relevant = q["relevant_doc_ids"]
    parts.append(f"### Query {qi + 1}: {_esc_cell(q['text'])}")
    parts.append(f"Gold-relevant doc ids: `{sorted(relevant)}`")
    parts.append("")
    parts.append(f"| mode | sanitized query | top-{DEMO_USE_TOP_K} retrieved | precision | recall |")
    parts.append("|---|---|---|---|---|")
    for m in USE_MODES:
        sanitized_q = queries_by_mode[m][qi]
        retrieved = retrievals[(qi, m)]
        if retrieved:
            hits = ", ".join(
                ("✓" if i in relevant else "✗") + f" doc {i}"
                for i in retrieved
            )
        else:
            hits = "(no matches)"
        retrieved_set = set(retrieved)
        tp = len(retrieved_set & relevant)
        precision = tp / len(retrieved_set) if retrieved_set else 0.0
        recall = tp / len(relevant) if relevant else 0.0
        parts.append(
            f"| `{m}` | `{_esc_cell(sanitized_q)}` | {hits} | {precision:.2f} | {recall:.2f} |"
        )
    parts.append("")

display(Markdown("\n".join(parts)))


## 5. Verdict

The retrieval tables tell the story. Average precision (across both queries, top-5):

| mode | avg precision | avg recall |
|---|---|---|
| `plain` (baseline) | 0.60 | 1.00 |
| `redact` | 0.40 | 0.625 |
| `label` | 0.40 | 0.625 |
| `label_number` | 0.30 | 0.50 |
| **`label_token`** | **0.60** | **1.00** |

**`label_token` is the only privacy-preserving mode that matches the no-privacy baseline.** Here's why each of the other three falls short, in plain terms:

- **`redact`** — the sanitized query is `********`. The sanitized version of *every* query for *any* PII value becomes the same string. Compare the top-5 returned by the alice query and the charlie query under `redact`: they're identical. The search has no way to tell what you were looking for.
- **`label`** — slightly better. The query is `[EMAIL]`, which at least distinguishes "looking for an email" from "looking for a phone." But the query is the same for `alice@example.com` and `charlie@example.com` — `[EMAIL]` either way — so the search still can't pick one over the other. Note: the `label` and `redact` retrievals look superficially similar, but `label`'s false positives are *other emails* while `redact` pulls in phone-only and address-only docs that aren't even close conceptually.
- **`label_number`** — same retrieval set as `label` (every email-bearing doc has an `[EMAIL_1]` token), with worse ranking. The per-doc counter resets, so a doc where alice is the second email tokenizes her as `[EMAIL_2]` — the query's `[EMAIL_1]` doesn't match her position there. Worse: the query's `[EMAIL_1]` *does* match other docs where someone unrelated (charlie, rachel, ops) happens to be the first email. Those become false positives by coincidence of position, not by intent.
- **`label_token`** — each `(label, plaintext)` pair gets one deterministic token from the Skyflow vault, identical across documents and queries. The alice query becomes `[EMAIL_u8UBDWQ]` and matches every doc where alice's email appears, regardless of whether she's the first or second email in that doc. It also doesn't match other emails. That's the property — referential integrity — that makes search (and joins, and RAG) actually work on sanitized text.

The Sanitization section showed *what* the four modes produce. This section shows *which of them are still usable downstream*.


# What's next?

## What this notebook covered

- **Detection** (sections 1–8): nine PII detectors benchmarked head-to-head on ai4privacy datasets. Skyflow wins overall F1, OPF is the strongest local option, GLiNER variants offer a tunable size-vs-quality trade-off, Presidio is the fastest but narrowest, ai4privacy and OpenMed under-perform their model cards on this benchmark.
- **Sanitization**: four ways to rewrite the spans the detectors find — `redact`, `label`, `label_number`, `label_token`. The first three are cheap and stateless; `label_token` requires a Skyflow vault but is the only mode whose output stays consistent across separate documents.
- **Use**: the mode you pick has real downstream consequences. Under BM25 search, `redact`, `label`, and `label_number` all reduce different PII queries to the same sanitized string and can't tell `alice@example.com` apart from `charlie@example.com`. Only `label_token` matches the no-privacy baseline on retrieval quality, because deterministic vault tokens preserve **referential integrity** — same plaintext, same token, every document.

## Suggested next steps

A natural follow-on is to swap the hand-curated demo data for **your own corpus**, then re-run all three pipelines against it:

1. **Build a JSONL fixture** in the same shape this harness already uses for benchmarking — one record per line, `{"text": "...", "gold_spans": [{"label": "EMAIL", "start": ..., "end": ...}, ...]}` (or just `{"text": "..."}` if you only want detection + sanitization, no scoring). [`opf_eval.fixtures.materialize`](https://github.com/jstjoe/local-privacy/blob/main/eval/src/opf_eval/fixtures.py) is the reference; mirror its output schema. Aim for representative coverage of the PII categories you actually care about (drop the ones you don't — the canonical taxonomy has 15 labels, you probably only need 4–6).
2. **Re-run detection (sections 1–8)** against the new fixture: same `runner.run(...)` call, point `--fixtures` at your file. F1 numbers will tell you which detector actually fits your data, not which one wins on a benchmark that may or may not look like your traffic.
3. **Re-run sanitization (sections under `# Sanitization`)** by setting `OUT_DIR` to the new run dir — the per-mode rendering is corpus-agnostic and pick up whatever `raw_<detector>.jsonl` files live there.
4. **Re-run the Use section** with `DEMO_USE_CORPUS` swapped for your real corpus and `DEMO_USE_QUERIES` swapped for queries that reflect how your users would search. Hand-define `relevant_doc_ids` (or use existing relevance judgments) and you get precision/recall per mode out of the box.
5. **Extend retrieval to RAG-style inference**. The Use section stops at retrieval; the next step is sending the retrieved sanitized docs into an LLM and scoring the generated answers. A RAGbench-style harness would loop: sanitize corpus → index → retrieve top-k per query → format prompt with retrieved chunks → generate answer → score against a gold answer. The interesting question is whether `label_token`'s referential integrity carries through to answer quality, and whether the LLM can reason over `[EMAIL_u8UBDWQ]`-style opaque tokens as fluently as over plaintext.
6. **Stop by the Skyflow booth.** We're demoing this notebook live — bring your own corpus or just questions about where deterministic tokenization fits into your privacy + retrieval stack. We're happy to walk through the setup, the `label_token` vault schema, and what production deployments of this look like.

The harness in this repo is intentionally narrow — it benchmarks one slice of one problem. Your own corpus is where the real questions are. Happy hacking!
